# Deliverable 3 · Evaluación Experimental de Premisas Falsas (P41–P50)

Este cuaderno ejecuta las **10 preguntas con premisas falsas / ausentes** del conjunto base
bajo las tres intervenciones de solución al bucle (*livelock*) del Auditor:

- **Opción 1**: Ajuste semántico del criterio de parada en `auditor.txt` (reconoce la ausencia normada como `aceptar`).
- **Opción 2**: Consenso semántico en el orquestador (`core.py`, si Redactor y Auditor coinciden en que no existe, aprueba).
- **Opción 3**: Flexibilización de apoyos para hechos negativos en `verificacion.txt`.

Total de inferencias: 10 preguntas (toma solo ~5-8 minutos en GPU T4 de Colab).

In [ ]:
# 1. Verificación de GPU CUDA
import torch
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Activa GPU T4 en: Entorno de ejecución > Cambiar tipo de entorno > GPU T4.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Instalación de dependencias fijadas
!pip install --quiet         transformers==4.57.6         accelerate==1.11.0         bitsandbytes==0.48.1         sentence-transformers==5.1.2         tokenizers==0.22.2

In [ ]:
# 3. Desempaquetado del entorno y las 10 preguntas
import base64, gzip, hashlib, json, shutil
from pathlib import Path

ARCHIVE = (
'...'
)
EXPECTED_SHA256 = 'ae62124d16bb488ea9afa17c4d73a44ec614858cb8752c80fd5237bb2606b20d'

data = gzip.decompress(base64.b64decode(ARCHIVE))
assert hashlib.sha256(gzip.compress(data, mtime=0)).hexdigest() == EXPECTED_SHA256
files = json.loads(data.decode('utf-8'))

ROOT = Path('workspace_p41_p50')
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)

for relpath, content in files.items():
    dest = ROOT / relpath
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(content, encoding='utf-8')

print(f"Entorno desempaquetado en {ROOT}. Total archivos: {len(files)}")

In [ ]:
# 4. Carga del Modelo Qwen3-4B y Retriever E5
import sys
sys.path.insert(0, str(Path("workspace_p41_p50").resolve()))

from d3_agentes.runtime import load
ENV = load("salida_runtime")
print("Modelo y retriever cargados exitosamente.")

In [ ]:
# 5. Ejecución experimental de las 10 preguntas con la Opción 1 (Ajuste sutil del Auditor)
from pathlib import Path
from d3_agentes.runtime import execute
from d3_agentes.experiment import export

print("Ejecutando las 10 preguntas de premisas falsas con el Auditor actualizado (Opción 1)...")
RUN_DIR = execute(ENV, ENV['questions'], Path("corrida_opcion1"), variants=['agentes_completo'], tag='p41-p50-opcion1', verbose=True)
zip_path = RUN_DIR.with_suffix('.zip')
print("Corrida completada y exportada a:", zip_path)

In [ ]:
# 6. Descarga del archivo ZIP de resultados
from google.colab import files
files.download(str(zip_path))